# 🏦 Credit Risk Predictor — Week 1: EDA & Feature Engineering

**Dataset:** Home Credit Default Risk (Kaggle)  
**Goal:** Understand the data, find patterns, engineer financial features

---
### Before running:
1. Upload `application_train.csv` to your Google Drive
2. Run **Cell 1** first to mount Drive and install packages
3. Then run cells one by one — read the output before moving on

In [ ]:
# ── CELL 1: Mount Google Drive & Install packages ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Install the one package Colab doesn't have by default
!pip install shap -q

print('✓ Drive mounted')
print('✓ Packages ready')

In [ ]:
# ── CELL 2: Set your file path ─────────────────────────────────────────────
# UPDATE THIS to wherever you put the file in Drive
# Example: if you put it in My Drive root:
DATA_PATH = '/content/drive/MyDrive/application_train.csv'

# Example: if you put it in a folder called 'credit-risk':
# DATA_PATH = '/content/drive/MyDrive/credit-risk/application_train.csv'

import os
if os.path.exists(DATA_PATH):
    size_mb = os.path.getsize(DATA_PATH) / 1024**2
    print(f'✓ File found! Size: {size_mb:.1f} MB')
else:
    print(f'✗ File NOT found at: {DATA_PATH}')
    print('  Check the path above and update DATA_PATH')

In [ ]:
# ── CELL 3: Imports ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('✓ All libraries loaded')

---
## Step 1 — Load & Inspect the Data

We load the CSV and do a quick shape + type check.
**What to look for:** number of rows, columns, and whether `TARGET` is present.

In [ ]:
# ── CELL 4: Load data ──────────────────────────────────────────────────────
print('Loading data... (may take 30–60 seconds)')
df = pd.read_csv(DATA_PATH)

print(f'\n Rows    : {df.shape[0]:,}')
print(f' Columns : {df.shape[1]}')
print(f'\n Column types:')
print(df.dtypes.value_counts().to_string())
print(f'\n First look at TARGET (our label):')
print(df['TARGET'].value_counts())

In [ ]:
# ── CELL 5: Peek at raw data ───────────────────────────────────────────────
# Shows first 3 rows — don't worry about all the columns,
# just notice the mix of numbers and text
df.head(3)

---
## Step 2 — Class Imbalance

**This is the most critical thing to understand before modelling.**

Real loan datasets are heavily imbalanced — most people repay. If we ignore this,
the model will just predict "no default" for everyone and get 92% accuracy while being useless.

We'll fix this in Week 2 using **SMOTE**.

In [ ]:
# ── CELL 6: Class imbalance plot ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['TARGET'].value_counts()

# Bar chart
bars = axes[0].bar(
    ['Non-Default (0)', 'Default (1)'],
    counts.values,
    color=['#4A90D9', '#E05C5C'],
    width=0.5,
    edgecolor='white'
)
axes[0].set_title('Loan Counts by Class', fontsize=13)
axes[0].set_ylabel('Number of Loans')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1000,
                 f'{val:,}', ha='center', fontsize=11)

# Pie chart
axes[1].pie(
    counts.values,
    labels=['Non-Default', 'Default'],
    colors=['#4A90D9', '#E05C5C'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11}
)
axes[1].set_title('Class Proportion', fontsize=13)

plt.suptitle('Target Variable — Loan Default', fontsize=14)
plt.tight_layout()
plt.show()

ratio = counts[0] / counts[1]
print(f'Imbalance ratio : {ratio:.1f}:1  (non-default : default)')
print(f'Default rate    : {counts[1]/counts.sum()*100:.2f}%')
print()
print('Insight: Only ~8% of loans defaulted.')
print('A naive model predicting "no default" every time would get 92% accuracy — but be completely useless!')
print('→ We will use SMOTE in Week 2 to balance the training set.')

---
## Step 3 — Missing Values

Real-world data is never clean. Some columns are almost entirely empty.

**Strategy:**
- Drop columns with **>40% missing** — not enough data to be useful
- Fill the rest with **median** (safe for skewed financial data)

In [ ]:
# ── CELL 7: Missing values summary ────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print(f'Total columns      : {df.shape[1]}')
print(f'Columns with nulls : {len(missing_df)}')
print(f'Columns >40% null  : {(missing_df.missing_pct > 40).sum()}  ← will be dropped')
print()
print('Top 15 by % missing:')
missing_df.head(15)

In [ ]:
# ── CELL 8: Missing values bar chart ──────────────────────────────────────
top25 = missing_df.head(25)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E05C5C' if p > 40 else '#4A90D9' for p in top25['missing_pct']]
ax.barh(top25.index[::-1], top25['missing_pct'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(x=40, color='orange', linestyle='--', linewidth=1.5, label='40% drop threshold')
ax.set_xlabel('% Missing')
ax.set_title('Top 25 Columns by Missing % (red = will be dropped)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 4 — Explore Key Financial Variables

Before engineering features, understand the raw distributions.
**Note:** `DAYS_BIRTH` and `DAYS_EMPLOYED` are stored as negative numbers (days before application date) — we convert them to positive years.

In [ ]:
# ── CELL 9: Distribution plots ─────────────────────────────────────────────
plot_config = {
    'AMT_INCOME_TOTAL' : ('Annual Income (₹)', False),
    'AMT_CREDIT'       : ('Loan Amount (₹)',   False),
    'AMT_ANNUITY'      : ('Annual Repayment',  False),
    'DAYS_BIRTH'       : ('Age (years)',        True),
    'DAYS_EMPLOYED'    : ('Employment (years)', True),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (col, (label, convert)) in enumerate(plot_config.items()):
    data = df[col].dropna().copy()
    if convert:
        data = data.replace(365243, np.nan).dropna()
        data = np.abs(data) / 365
    # Cap extreme outliers for cleaner plots
    cap = data.quantile(0.99)
    data = data[data <= cap]
    axes[i].hist(data, bins=50, color='#4A90D9', alpha=0.75, edgecolor='white')
    axes[i].set_title(label, fontsize=11)
    axes[i].set_ylabel('Count')
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'
    ))

axes[-1].set_visible(False)
plt.suptitle('Distribution of Key Financial Variables', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 10: Default rate by category ─────────────────────────────────────
# Very useful insight — which groups default more?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By education
edu = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
edu.plot(kind='barh', ax=axes[0], color='#E05C5C', alpha=0.8, edgecolor='white')
axes[0].set_title('Default Rate by Education Level', fontsize=12)
axes[0].set_xlabel('Default Rate (%)')
for i, v in enumerate(edu.values):
    axes[0].text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=10)

# By income type
inc = df.groupby('NAME_INCOME_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
inc.plot(kind='barh', ax=axes[1], color='#4A90D9', alpha=0.8, edgecolor='white')
axes[1].set_title('Default Rate by Income Type', fontsize=12)
axes[1].set_xlabel('Default Rate (%)')
for i, v in enumerate(inc.values):
    axes[1].text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()
print('Insight: Education and income type are strong default predictors — these are captured in our features.')

In [ ]:
# ── CELL 11: What raw features correlate with default? ─────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()['TARGET'].drop('TARGET').abs().sort_values(ascending=False)

top15 = corr.head(15)
fig, ax = plt.subplots(figsize=(9, 5))
top15[::-1].plot(kind='barh', ax=ax, color='#4A90D9', alpha=0.8, edgecolor='white')
ax.set_title('Top 15 Raw Features Correlated with Default', fontsize=13)
ax.set_xlabel('|Correlation with TARGET|')
plt.tight_layout()
plt.show()

print('Top 5 raw features:')
for feat, val in top15.head(5).items():
    print(f'  {feat:<35} {val:.4f}')
print()
print('Insight: EXT_SOURCE_2 and EXT_SOURCE_3 (external credit scores) are the strongest signals.')
print('We will engineer features that combine all 3 EXT_SOURCE scores.')

---
## Step 5 — Feature Engineering ⭐

This is the most important part of Week 1.

Raw columns like `AMT_ANNUITY` and `AMT_INCOME_TOTAL` are useful but not as powerful as their **ratio** — the Debt-to-Income ratio tells you how much of someone's income goes to loan repayment.

We engineer **12 domain-driven features** that a credit analyst would actually look at.

In [ ]:
# ── CELL 12: Feature engineering function ──────────────────────────────────
def engineer_features(df):
    data = df.copy()

    # 1. Debt-to-Income — repayment as fraction of income
    #    High DTI = risky (paying too much of income on loan)
    data['DEBT_TO_INCOME'] = data['AMT_ANNUITY'] / (data['AMT_INCOME_TOTAL'] + 1)

    # 2. Credit-to-Income — total loan vs income
    data['CREDIT_TO_INCOME'] = data['AMT_CREDIT'] / (data['AMT_INCOME_TOTAL'] + 1)

    # 3. Annuity-to-Credit — proxy for loan term length
    #    High = shorter loan (aggressive repayment)
    data['ANNUITY_TO_CREDIT'] = data['AMT_ANNUITY'] / (data['AMT_CREDIT'] + 1)

    # 4. Age in years — DAYS_BIRTH is negative (days before application)
    data['AGE_YEARS'] = np.abs(data['DAYS_BIRTH']) / 365

    # 5. Employment years — 365243 is a special code meaning unemployed/pensioner
    data['DAYS_EMPLOYED'] = data['DAYS_EMPLOYED'].replace(365243, np.nan)
    data['EMPLOYMENT_YEARS'] = np.abs(data['DAYS_EMPLOYED']) / 365

    # 6. Employment-to-Age ratio — job stability signal
    #    High = spent most of working life employed (stable)
    data['EMPLOYMENT_TO_AGE'] = data['EMPLOYMENT_YEARS'] / (data['AGE_YEARS'] + 1)

    # 7. Income per family member — effective disposable income
    data['INCOME_PER_PERSON'] = data['AMT_INCOME_TOTAL'] / (data['CNT_FAM_MEMBERS'] + 1)

    # 8. Loan-to-Value — how much of goods price is being financed
    if 'AMT_GOODS_PRICE' in data.columns:
        data['LOAN_TO_VALUE'] = data['AMT_CREDIT'] / (data['AMT_GOODS_PRICE'] + 1)

    # 9. External credit score aggregates
    #    Home Credit gives 3 scores from external bureaus — combine them
    ext = [c for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'] if c in data.columns]
    if ext:
        data['EXT_SOURCE_MEAN'] = data[ext].mean(axis=1)
        data['EXT_SOURCE_MIN']  = data[ext].min(axis=1)   # worst score

    # 10. Document count — submitting more docs = more creditworthy
    doc_cols = [c for c in data.columns if 'FLAG_DOCUMENT' in c]
    data['DOCS_SUBMITTED'] = data[doc_cols].sum(axis=1) if doc_cols else 0

    # 11. Young applicant flag — under 27 = statistically higher default rate
    data['IS_YOUNG_APPLICANT'] = (data['AGE_YEARS'] < 27).astype(int)

    # 12. Registration recency
    data['REG_YEARS'] = np.abs(data['DAYS_REGISTRATION']) / 365

    return data


print('Engineering features...')
df_feat = engineer_features(df)

new_features = ['DEBT_TO_INCOME','CREDIT_TO_INCOME','ANNUITY_TO_CREDIT',
                'AGE_YEARS','EMPLOYMENT_YEARS','EMPLOYMENT_TO_AGE',
                'INCOME_PER_PERSON','LOAN_TO_VALUE','EXT_SOURCE_MEAN',
                'EXT_SOURCE_MIN','DOCS_SUBMITTED','IS_YOUNG_APPLICANT']

print(f'\n✓ {len(new_features)} new features created')
print(f'  Dataset shape: {df.shape} → {df_feat.shape}')

In [ ]:
# ── CELL 13: Do our new features actually predict default? ─────────────────
# Sanity check — if engineered features don't correlate with TARGET,
# we wasted our time. Let's verify.

available = [f for f in new_features if f in df_feat.columns]
eng_corr = (
    df_feat[available + ['TARGET']]
    .corr()['TARGET']
    .drop('TARGET')
    .abs()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#5BAD8F' if v > 0.05 else '#AAAAAA' for v in eng_corr.values]
eng_corr.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(y=0.05, color='orange', linestyle='--', linewidth=1, label='0.05 threshold (useful signal)')
ax.set_title('Engineered Features — Correlation with Default', fontsize=13)
ax.set_ylabel('|Correlation with TARGET|')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

print('Feature correlations with default:')
for f, v in eng_corr.items():
    mark = '✓' if v > 0.05 else '~'
    print(f'  {mark} {f:<25} {v:.4f}')

---
## Step 6 — Clean & Save

Final cleaning pipeline before saving:
1. Drop high-missing columns
2. Encode text columns to numbers
3. Fill remaining nulls with median
4. Save to Drive for Week 2

In [ ]:
# ── CELL 14: Clean dataset ─────────────────────────────────────────────────
def clean_dataset(df, drop_threshold=0.40):
    data = df.copy()

    # Drop ID (not a feature)
    data = data.drop(columns=['SK_ID_CURR'], errors='ignore')

    # Drop high-missing columns
    missing_pct = data.isnull().mean()
    drop_cols = missing_pct[missing_pct > drop_threshold].index.tolist()
    data = data.drop(columns=drop_cols)
    print(f'Dropped {len(drop_cols)} columns with >{drop_threshold*100:.0f}% missing')

    # Encode categorical columns
    cat_cols = data.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        data[col] = pd.Categorical(data[col]).codes
    print(f'Encoded {len(cat_cols)} categorical columns')

    # Fill numeric nulls with median
    num_cols = data.select_dtypes(include=[np.number]).columns
    nulls_before = data[num_cols].isnull().sum().sum()
    data[num_cols] = data[num_cols].fillna(data[num_cols].median())
    print(f'Filled {nulls_before:,} remaining null values with median')

    # Remove infinities
    data = data.replace([np.inf, -np.inf], np.nan).fillna(0)

    print(f'\nFinal shape  : {data.shape}')
    print(f'Null values  : {data.isnull().sum().sum()}')
    return data


df_clean = clean_dataset(df_feat)
df_clean.head(3)

In [ ]:
# ── CELL 15: Save processed dataset to Drive ───────────────────────────────
# UPDATE this path if you want to save somewhere specific in your Drive
SAVE_PATH = '/content/drive/MyDrive/train_processed.csv'

df_clean.to_csv(SAVE_PATH, index=False)

size_mb = os.path.getsize(SAVE_PATH) / 1024**2
print(f'✓ Saved to: {SAVE_PATH}')
print(f'  Rows    : {df_clean.shape[0]:,}')
print(f'  Columns : {df_clean.shape[1]}')
print(f'  Size    : {size_mb:.1f} MB')
print()
print('This file is your input for Week 2 model training.')

---
## ✅ Week 1 Complete!

| What you did | Key finding |
|---|---|
| Loaded 307K loan records | 122 raw columns |
| Found class imbalance | Only 8% defaulted → need SMOTE |
| Handled missing values | Dropped high-null cols, imputed rest |
| Explored distributions | Age, income, loan amounts all skewed |
| Engineered 12 features | DTI ratio, EXT_SOURCE_MEAN strongest |
| Saved clean dataset | Ready for Week 2 |

---
### Git commits to make now:
```
git add notebooks/week1_colab.ipynb
git commit -m "feat: add EDA notebook with class imbalance analysis"

git add notebooks/week1_colab.ipynb
git commit -m "feat: engineer 12 domain-driven financial features"
```

### Next: Week 2 — Model Training
- Logistic Regression baseline
- XGBoost with SMOTE
- AUC-ROC evaluation